# Universal policy simulation on the actual main-phase maps

This notebook keeps the gameplay environment separate from the model calculation:

- `main_phase_universal_sim.py`: actual map/gameplay environment ported from `js/main_phase.js`
- `universal_policy_model.py`: universal policy calculation and tunable model parameters

To twist the model, edit the parameter code block below, then run the simulation cell. The animation shows the game board on the left and the action stream on the right. `FRAME_MS` controls how long each action frame stays on screen.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next((p for p in candidates if (p / "js" / "main_phase.js").exists()), candidates[0])
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from universal_model_playground.universal_policy_model import UniversalPolicyParams
from universal_model_playground.main_phase_universal_sim import (
    available_main_phase_maps,
    display_simulation,
    simulate_universal_pair,
)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
maps = available_main_phase_maps(REPO_ROOT)
for i, path in enumerate(maps):
    print(f"{i}: {path.name}")

## Twist parameters here

Edit this cell before running the simulation. `FRAME_MS` is the per-action display time in milliseconds, and `ACTION_ROWS` controls how many recent actions are visible in the side panel.

In [ ]:
MAP_INDEX = 0
SEED = 7
ROUNDS = 15
MAX_MOVES_PER_TURN = 5

FRAME_MS = 320       # time each action frame stays on screen
ACTION_ROWS = 18     # number of action rows shown on the side
MAX_FRAMES = None    # set to an integer for a shorter preview

FORAGER_MODEL = UniversalPolicyParams(
    lambda_value=0.00,
    info_reward_tradeoff=0.05,
    epsilon=20.0,
    beta=0.25,
    chase_wt_drop=0.25,
    vdig_vmove_tradeoff=0.70,
)

SECURITY_MODEL = UniversalPolicyParams(
    lambda_value=0.00,
    info_reward_tradeoff=0.05,
    epsilon=20.0,
    beta=0.25,
    chase_wt_drop=0.25,
    vdig_vmove_tradeoff=0.70,
)

## Run animated simulation

In [ ]:
maps = available_main_phase_maps(REPO_ROOT)
selected_map = maps[MAP_INDEX]

result = simulate_universal_pair(
    selected_map,
    forager_params=FORAGER_MODEL,
    security_params=SECURITY_MODEL,
    rounds=ROUNDS,
    max_moves=MAX_MOVES_PER_TURN,
    seed=SEED,
    repo_root=REPO_ROOT,
)

print(f"Map: {selected_map.name}")
print(f"Final gold: {result.state.gold_total}")
print(f"Frames: {len(result.frames)} | Events: {len(result.events)}")

display_simulation(result, frame_ms=FRAME_MS, action_rows=ACTION_ROWS, max_frames=MAX_FRAMES)
display(result.events_dataframe().tail(20))